In [1]:
# 1. Install crawl4ai and nest_asyncio (needed for Colab's event loop)
%%capture
!pip install -U crawl4ai nest_asyncio pydantic -q

# 2. Set up Crawl4AI's virtual browser environment
!crawl4ai-setup

# 3. Verify the installation is correct
!crawl4ai-doctor

### Step 2: The Gemini Scraping Script
This script prompts you securely for your Gemini API key, defines a target schema (e.g., extracting course names and descriptions from a website), and uses Gemini to intelligently structure the data.

#### Scrapping Website **Books To scrape**

In [2]:
import os
import json
import asyncio
import nest_asyncio
from getpass import getpass
from pydantic import BaseModel, Field
from typing import List, Type
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig, CacheMode, LLMConfig
from crawl4ai.extraction_strategy import LLMExtractionStrategy

# Ensure nest_asyncio is applied for Colab
nest_asyncio.apply()

# Securely get your Gemini API key from the user if not already set
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")

async def scrape_with_llm_extraction(
    target_url: str,
    schema_model: Type[BaseModel],
    instruction: str,
    llm_provider: str = "gemini/gemini-2.5-flash"
):
    """
    A function to scrape a website using LLM-based structured data extraction.

    Args:
        target_url (str): The URL of the website to scrape.
        schema_model (Type[BaseModel]): The Pydantic BaseModel defining the desired output structure.
        instruction (str): The natural language instruction for the LLM to guide extraction.
        llm_provider (str): The LLM provider and model name (e.g., "gemini/gemini-2.5-flash").

    Returns:
        dict: The extracted structured data as a dictionary, or None if extraction fails.
    """
    print(f"\nCrawling and analyzing: {target_url} ... Please wait ...")

    llm_config = LLMConfig(
        provider=llm_provider,
        api_token=os.environ["GEMINI_API_KEY"]
    )

    extraction_strategy = LLMExtractionStrategy(
        llm_config=llm_config,
        schema=schema_model.model_json_schema(),
        extraction_type="schema",
        instruction=instruction
    )

    run_config = CrawlerRunConfig(
        extraction_strategy=extraction_strategy,
        cache_mode=CacheMode.BYPASS # Force a fresh crawl
    )

    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(
            url=target_url,
            config=run_config
        )

        if result.success:
            print("\n✅ Extraction Successful!")
            structured_data = json.loads(result.extracted_content)
            print("\n--- Extracted Structured Data ---")
            print(json.dumps(structured_data, indent=2))
            return structured_data
        else:
            print(f"\n❌ Crawl Failed. Error: {result.error_message}")
            return None


# --- Example Usage for a Different Website and Schema ---

# 1. Define a new Pydantic Schema for books
class Book(BaseModel):
    title: str = Field(..., description="The title of the book.")
    author: str = Field(..., description="The author of the book.")
    price: str = Field(..., description="The price of the book, including currency.")
    availability: str = Field(..., description="The availability status of the book (e.g., 'In stock', 'Out of stock').")

class BookList(BaseModel):
    books: List[Book] = Field(..., description="A list of books extracted from the page.")

# 2. Set your target URL and instruction
book_url = "https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html"
book_instruction = "Extract book titles, authors, prices, and availability from the page."

# 3. Run the generalized scraping function
await scrape_with_llm_extraction(book_url, BookList, book_instruction)


Enter your Gemini API Key: ··········

Crawling and analyzing: https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html ... Please wait ...


[INIT].... → Crawl4AI 0.9.2 

[FETCH]... ↓ https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html                            | ✓ | ⏱: 1.39s 

[SCRAPE].. ◆ https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html                            | ✓ | ⏱: 0.10s 

[EXTRACT]. ■ https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html                            | ✓ | ⏱: 21.45s 

[COMPLETE] ● https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html                            | ✓ | ⏱: 22.97s 


✅ Extraction Successful!

--- Extracted Structured Data ---
[
  {
    "books": [
      {
        "title": "Unicorn Tracks",
        "author": "",
        "price": "\u00a318.78",
        "availability": "In stock"
      },
      {
        "title": "Saga, Volume 6 (Saga (Collected Editions) #6)",
        "author": "",
        "price": "\u00a325.02",
        "availability": "In stock"
      },
      {
        "title": "Princess Between Worlds (Wide-Awake Princess #5)",
        "author": "",
        "price": "\u00a313.34",
        "availability": "In stock"
      },
      {
        "title": "Masks and Shadows",
        "author": "",
        "price": "\u00a356.40",
        "availability": "In stock"
      },
      {
        "title": "Crown of Midnight (Throne of Glass #2)",
        "author": "",
        "price": "\u00a343.29",
        "availability": "In stock"
      },
      {
        "title": "Avatar: The Last Airbender: Smoke and Shadow, Part 3 (Smoke and Shadow #3)",
        "author": 

[{'books': [{'title': 'Unicorn Tracks',
    'author': '',
    'price': '£18.78',
    'availability': 'In stock'},
   {'title': 'Saga, Volume 6 (Saga (Collected Editions) #6)',
    'author': '',
    'price': '£25.02',
    'availability': 'In stock'},
   {'title': 'Princess Between Worlds (Wide-Awake Princess #5)',
    'author': '',
    'price': '£13.34',
    'availability': 'In stock'},
   {'title': 'Masks and Shadows',
    'author': '',
    'price': '£56.40',
    'availability': 'In stock'},
   {'title': 'Crown of Midnight (Throne of Glass #2)',
    'author': '',
    'price': '£43.29',
    'availability': 'In stock'},
   {'title': 'Avatar: The Last Airbender: Smoke and Shadow, Part 3 (Smoke and Shadow #3)',
    'author': '',
    'price': '£28.09',
    'availability': 'In stock'},
   {'title': 'A Court of Thorns and Roses (A Court of Thorns and Roses #1)',
    'author': '',
    'price': '£52.37',
    'availability': 'In stock'},
   {'title': 'Throne of Glass (Throne of Glass #1)',
    '

#### Scrapping Website **KidoCode**

In [ ]:
import os
import json
import asyncio
import nest_asyncio
from getpass import getpass
from pydantic import BaseModel, Field
from typing import List
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig, CacheMode, LLMConfig
from crawl4ai.extraction_strategy import LLMExtractionStrategy

# 1. Allow nested async loops inside Google Colab
nest_asyncio.apply()

# 2. Securely get your Gemini API key from the user
# You can get a free key from Google AI Studio (https://aistudio.google.com/)
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")

# 3. Define the structure you want the AI to return using Pydantic
class Course(BaseModel):
    title: str = Field(..., description="The title of the course or subject.")
    difficulty: str = Field(..., description="The difficulty level if mentioned (e.g., Beginner, Intermediate, Advanced).")
    description: str = Field(..., description="A brief summary of what the course covers.")

class CourseList(BaseModel):
    courses: List[Course] = Field(..., description="A list of courses extracted from the page.")

async def extract_structured_data():
    # 4. Configure the LLM Strategy to use Gemini
    # Crawl4AI integrates with LiteLLM. Gemini models use the 'gemini/' prefix.
    # We use 'gemini-2.5-flash' as it is fast, highly capable, and has a generous free tier.
    llm_config = LLMConfig(
        provider="gemini/gemini-2.5-flash",
        api_token=os.environ["GEMINI_API_KEY"]
    )

    extraction_strategy = LLMExtractionStrategy(
        llm_config=llm_config,
        schema=CourseList.model_json_schema(), # Feed the Pydantic schema to the LLM
        extraction_type="schema",
        instruction="Extract technologhy courses listed on the page with their titles, difficulty levels, and a summary description."
    )

    # 5. Create Crawler Configuration
    run_config = CrawlerRunConfig(
        extraction_strategy=extraction_strategy,
        cache_mode=CacheMode.BYPASS # Force a fresh crawl
    )

    # 6. Run the crawler
    # We are using a sample educational page, but you can change this to any URL!
    target_url = "https://www.kidocode.com/degrees/technology"

    print(f"Crawling and analyzing: {target_url} ... Please wait ...")

    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(
            url=target_url,
            config=run_config
        )

        # 7. Print results
        if result.success:
            print("\n✅ Extraction Successful!")
            # The structured JSON is stored in result.extracted_content
            structured_data = json.loads(result.extracted_content)
            print("\n--- Extracted Structured Data ---")
            print(json.dumps(structured_data, indent=2))

        else:
            print(f"\n❌ Crawl Failed. Error: {result.error_message}")

# Execute the asynchronous crawler inside Colab
await extract_structured_data()

Crawling and analyzing: https://www.kidocode.com/degrees/technology ... Please wait ...


[INIT].... → Crawl4AI 0.9.2 

[FETCH]... ↓ https://www.kidocode.com/degrees/technology                                                          | ✓ | ⏱: 2.40s 

[SCRAPE].. ◆ https://www.kidocode.com/degrees/technology                                                          | ✓ | ⏱: 0.13s 

[EXTRACT]. ■ https://www.kidocode.com/degrees/technology                                                          | ✓ | ⏱: 19.27s 

[COMPLETE] ● https://www.kidocode.com/degrees/technology                                                          | ✓ | ⏱: 21.85s 


✅ Extraction Successful!

--- Extracted Structured Data ---
[
  {
    "courses": [
      {
        "title": "Coding with Python",
        "difficulty": "Beginner",
        "description": "Explore the exciting intersection of art and technology in Creative Coding with Python. This engaging course introduces young learners to Python programming, teaching them foundational coding concepts through the creation of generative art and geometric patterns. As students progress from basic programming principles to artistic project development, they'll enhance their problem-solving and logical thinking abilities, all within a creative and fun learning environment."
      },
      {
        "title": "Electronics & Robotics",
        "difficulty": "Beginner",
        "description": "Electronics and Robotics is a foundational course designed to immerse you in the fundamentals of electronics and robotics. Start with an introduction to key concepts and progress to understanding the roles of essential

### Python Implementation of a Dynamic Scraper
It takes a raw user query, dynamically designs the JSON schema, and executes LLMExtractionStrategy without any pre-defined Python models:  

In [10]:
import os
import json
import asyncio
import nest_asyncio
from google import genai
from google.genai import types
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig, CacheMode, LLMConfig
from crawl4ai.extraction_strategy import LLMExtractionStrategy

nest_asyncio.apply()

# Initialize Gemini Client (using the official Google GenAI SDK)
# Ensure os.environ["GEMINI_API_KEY"] is set before running
client = genai.Client()

# ==========================================
# STEP 1: DYNAMIC JSON SCHEMA GENERATOR
# ==========================================
def generate_dynamic_schema(user_query: str) -> dict:
    """
    Uses Gemini to look at the user's plain English request and
    returns a raw JSON Schema (Draft 7/Pydantic equivalent dictionary).
    """
    print(f"🤖 AI Agent: Designing a custom schema for: '{user_query}'...")

    prompt = f"""
    Analyze the user's data extraction request: "{user_query}"

    Generate a JSON Schema (dictionary format) that can represent this data.
    The outer object MUST be a container/wrapper (e.g., 'items' or 'products' or 'results')
    which holds a list of the extracted entities.

    Return ONLY a valid JSON object. No markdown, no wrap.
    """

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        )
    )

    # Parse the returned string into a Python dict
    schema_dict = json.loads(response.text)
    return schema_dict

# ==========================================
# STEP 2: RUN DYNAMIC CRAWL4AI SCRAPER
# ==========================================
async def scrape_any_website(target_url: str, user_query: str):
    # 1. Generate the custom schema on-the-fly
    dynamic_schema = generate_dynamic_schema(user_query)

    print("\n📝 Generated Dynamic Schema:")
    print(json.dumps(dynamic_schema, indent=2))

    # 2. Feed the dynamic schema dictionary directly into Crawl4AI
    llm_config = LLMConfig(
        provider="gemini/gemini-2.5-flash", # This part is for crawl4ai and might have different model names
        api_token=os.environ.get("GEMINI_API_KEY")
    )

    # Passing dynamic_schema dict to schema parameter (no Pydantic class required!)
    extraction_strategy = LLMExtractionStrategy(
        llm_config=llm_config,
        schema=dynamic_schema,
        extraction_type="schema",
        instruction=f"Extract data matching the user request: {user_query}"
    )

    run_config = CrawlerRunConfig(
        extraction_strategy=extraction_strategy,
        cache_mode=CacheMode.BYPASS
    )

    print(f"\n🕷️ Crawl4AI: Scraping {target_url} using the dynamic strategy...")
    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(url=target_url, config=run_config)

        if result.success:
            print("\n🎉 Extraction Successful!")
            print(json.dumps(json.loads(result.extracted_content), indent=2))
        else:
            print(f"❌ Scraping failed: {result.error_message}")

# ==========================================
# EXAMPLE EXECUTION
# ==========================================
# The user wants to scrape a cryptocurrency page
user_url = "https://coinmarketcap.com/"
query = "Extract the top cryptos listing showing their Name, Price, and 24h % Change"

await scrape_any_website(user_url, query)



--- Debugging: Listing available Gemini models ---
Error listing models: module 'google.genai' has no attribute 'list_models'. Please ensure your GEMINI_API_KEY is correctly set and has network access.
🤖 AI Agent: Designing a custom schema for: 'Extract the top cryptos listing showing their Name, Price, and 24h % Change'...

📝 Generated Dynamic Schema:
{
  "type": "object",
  "properties": {
    "cryptos": {
      "type": "array",
      "description": "List of top cryptocurrencies with their details.",
      "items": {
        "type": "object",
        "properties": {
          "Name": {
            "type": "string",
            "description": "The name of the cryptocurrency."
          },
          "Price": {
            "type": "number",
            "description": "The current price of the cryptocurrency."
          },
          "24h % Change": {
            "type": "number",
            "description": "The percentage change in price over the last 24 hours."
          }
        },
 

[INIT].... → Crawl4AI 0.9.2 

[FETCH]... ↓ https://coinmarketcap.com/                                                                           | ✓ | ⏱: 16.58s 

[SCRAPE].. ◆ https://coinmarketcap.com/                                                                           | ✓ | ⏱: 1.91s 

[EXTRACT]. ■ https://coinmarketcap.com/                                                                           | ✓ | ⏱: 13.99s 

[COMPLETE] ● https://coinmarketcap.com/                                                                           | ✓ | ⏱: 32.75s 


🎉 Extraction Successful!
[
  {
    "cryptos": [
      {
        "Name": "CoinMarketCap 20 Index",
        "Price": 131.33,
        "24h % Change": 2.47
      },
      {
        "Name": "Bitcoin",
        "Price": 65001.78,
        "24h % Change": 1.99
      },
      {
        "Name": "Ethereum",
        "Price": 1912.15,
        "24h % Change": 3.66
      },
      {
        "Name": "Tether",
        "Price": 0.9991,
        "24h % Change": 0.02
      },
      {
        "Name": "BNB",
        "Price": 578.63,
        "24h % Change": 0.77
      },
      {
        "Name": "USDC",
        "Price": 0.9998,
        "24h % Change": 0.0
      },
      {
        "Name": "XRP",
        "Price": 1.11,
        "24h % Change": 2.53
      },
      {
        "Name": "Solana",
        "Price": 77.54,
        "24h % Change": 2.03
      },
      {
        "Name": "TRON",
        "Price": 0.3279,
        "24h % Change": 0.8
      },
      {
        "Name": "Hyperliquid",
        "Price": 68.27,
        